# A complete day

§5 of the problem definition decomposes the operation into five stages, and
§5.6 runs them as one day. This notebook is that day, end to end, using the
same functions the API and the schedulers call.

**Every number here is provisional.** The outcome rates are read off §10's
worked example and the processing throughputs are invented — `docs/assumptions.md`
says so per row. A delivered total from this notebook restates the document; it
does not measure the operation. `docs/capacity-finding.md` is the standing
warning about the difference.

The one thing it *does* demonstrate is that the stages hand work to each other
without losing an envelope.

## The shape of a day

§3.1's one-day lag is why a day does two unrelated things at once:

- it **delivers** the pool positioned yesterday (§5.4, then §6's outcomes, then §5.5's return run), and
- it **collects and positions** the pool that will be delivered tomorrow (§5.1 → §5.2 → §5.3).

They share a date and a van fleet and nothing else.

In [ ]:
from datetime import datetime

from ddn import assumptions
from ddn.model import travel as road
from ddn.simulation import State, render, run_day, run_days
from tests.fixtures import peak_day
from tests.matrices import road_matrix

HOUR = 3600
day = peak_day.load()
print(f"collection day {day.collection_day}  ->  delivery day {day.delivery_day}")
print(f"{len(day.ready()):,} envelopes positioned today for tomorrow")
print(f"{len(peak_day.morning_pool()):,} in this morning's pool, from yesterday")

## The inputs

§9.1 records, in the shapes the modules take. This is the only verbose part of
the notebook; the fixture is §10's worked example, encoded.

In [ ]:
midnight = datetime.combine(day.collection_day, datetime.min.time())
secs = lambda moment: int((moment - midnight).total_seconds())

facilities = [
    {"id": f.facility_id, "lat": f.lat, "lon": f.lon,
     "route_release_time": f.route_release_time.hour * HOUR,
     "transit_from_hub_min": f.transit_from_hub_min,
     "shift_start": 7 * HOUR, "shift_end": 15 * HOUR}
    for f in day.facilities
]

pools = {}
for e in peak_day.morning_pool():
    pools.setdefault(e.facility_id, []).append({
        "package_id": e.package_id, "customer_id": e.customer_id,
        "lat": e.lat, "lon": e.lon, "status": "Ready",
        "geocode_confidence": "high", "priority": float(e.priority),
        "sla_date": e.sla_date.isoformat(), "facility_id": e.facility_id,
        "attempt_number": e.attempt_number,
        "customer_lat": e.lat, "customer_lon": e.lon,
    })

requests = [
    {"mailbag_id": b.mailbag_id, "customer_id": b.customer_id,
     "lat": r.lat, "lon": r.lon, "requested_at": secs(r.requested_at),
     "expected_weight_g": b.expected_weight_g,
     "envelope_count": b.envelope_count}
    for r in day.requests for b in r.mailbags
]
inflow = [
    {"package_id": e.package_id, "mailbag_id": e.mailbag_id,
     "package_type": e.package_type, "facility_id": e.facility_id,
     "lat": e.lat, "lon": e.lon, "status": "Ready",
     "geocode_confidence": "high", "priority": float(e.priority),
     "sla_date": e.sla_date.isoformat(), "customer_id": e.customer_id}
    for e in day.ready()
]
vans = [
    {"vehicle_id": v.vehicle_id, "type": "van", "role": v.role.value,
     "capacity_mailbags": v.capacity_mailbags,
     "capacity_weight_g": v.capacity_weight_g,
     "shift_start": secs(v.shift_start), "shift_end": secs(v.shift_end),
     "linehaul_release_at": (None if v.linehaul_release_at is None
                             else secs(v.linehaul_release_at))}
    for v in day.vehicles if v.type.value == "van"
]
bikes = [v.vehicle_id for v in day.vehicles if v.type.value == "motorbike"]

print(f"{len(facilities)} facilities, {len(bikes)} motorbikes, {len(vans)} vans")
print(f"{len(requests)} mailbags carrying {len(inflow):,} envelopes")

## Travel

§3.3 assigns by road distance and §5.1 routes on it, so `run_day` takes travel from a matrix rather than a speed.

These are **real road distances**, recorded once against OSRM over the Costa Rica extract and replayed from `tests/fixtures/road.json.gz` — so this notebook routes on roads without needing a gateway. The *network* is real; the *places* are the fixture's synthetic ones, because §3.1 still does not supply the operation's geography.

It matters more than it sounds. Hub to D1 is 11,551 m by road against 8,235 m as the crow flies: a factor of 1.40, applied to every leg of every pickup route.

In [ ]:
# §5.1's legs run between the hub and the bag sites, so the matrix
# spans exactly those. A fake — see tests/matrices.py.
points = [facilities[0], *requests]
matrix = road_matrix(points)
travel = road.over(matrix, road.index_of(points))

travel(facilities[0]["lat"], facilities[0]["lon"],
       requests[0]["lat"], requests[0]["lon"]), "seconds, hub to the first site"

## One day

`run_day` takes yesterday's hand-over and returns the day's report plus the
state it hands tomorrow. `allocation` is passed because §10 states its own
motorbike split; without it, §4.2 derives one from the pool.

In [ ]:
state = State(day=day.delivery_day, pools={f: tuple(p) for f, p in pools.items()})

report, tomorrow = run_day(
    state,
    facilities=facilities, bikes=bikes, vans=vans,
    requests=requests, inflow=inflow, travel=travel,
    allocation=dict(peak_day.BIKE_ALLOCATION),
    seed=7,
)
print(render(report))

## What the day handed on

§8.3 expects a structural gap, and §10 ends by showing one. Nothing is lost
between the stages: every envelope is delivered, held, returned or carried.

In [ ]:
capacity = len(bikes) * assumptions.ENVELOPES_PER_BIKE
print(f"tomorrow's pool   {tomorrow.pool_size:,}")
print(f"fleet can serve   {capacity:,}")
print(f"short by          {tomorrow.pool_size - capacity:,}")
print()
print("accounted for today:")
t = report.tally
print(f"  delivered {t.delivered:,}  postponed {t.postponed:,}  "
      f"rejected {t.rejected:,}  defective {t.defective:,}  "
      f"unassigned {t.unassigned:,}")
assert t.delivered + t.postponed + t.rejected + t.defective == t.dispatched
print("\n  dispatched == the four outcomes, exactly")

## Several days

One day hides a structural gap; §8.3 expects it to accumulate. `run_days`
chains the hand-over so the backlog is visible.

In [ ]:
week = run_days(3, state, facilities=facilities, bikes=bikes, vans=vans,
                requests=requests, inflow=inflow, travel=travel,
                allocation=dict(peak_day.BIKE_ALLOCATION), seed=7)

print(f"{'day':<12}{'pool':>9}{'delivered':>11}{'unassigned':>12}{'binds':>12}")
for r in week:
    print(f"{r.day.isoformat():<12}{r.tally.ready_pool:>9,}"
          f"{r.tally.delivered:>11,}{r.tally.unassigned:>12,}"
          f"{(r.checks.binding.name if r.checks.binding else '—'):>12}")

The morning pool grows every day and the unassigned count grows with it. That
is §8.3's gap, and no solver setting closes it — it is a fleet too small for the
inflow, which is a decision for operations rather than for routing.

## Read that table again

Day three reports more delivered than the fleet can carry. That is not a bug,
and it is the most important thing in this notebook.

In [ ]:
capacity = len(bikes) * assumptions.ENVELOPES_PER_BIKE
print(f"{'day':<12}{'delivered':>11}{'fleet can serve':>18}{'due that day':>14}")
for r in week:
    due = sum(1 for pool in (inflow,) for e in pool
              if e["sla_date"] == r.day.isoformat())
    flag = "  <-- more than the fleet" if r.tally.delivered > capacity else ""
    print(f"{r.day.isoformat():<12}{r.tally.delivered:>11,}{capacity:>18,}"
          f"{due:>14,}{flag}")

Two rules meet here and the result is a figure that must not be read as a
capacity measurement.

**§6.1 makes "SLA date = today" a hard constraint, not a weight** — an envelope
due today is never trimmed by `lastmile.select`, whatever the facility's bike
count. The document says "subject to feasibility", and feasibility is the
solver's half of the sentence.

**This day has no solver.** `run_day`'s default `deliver` attempts everything
selection offers, because selection has already cut the pool to what the bikes
could carry — *except* for the envelopes §6.1 will not let it cut. On day three
most of the positioned pool comes due at once, so they are all forced through
and nothing is left to refuse them.

So a default run is the **optimistic bound**: what would be delivered if travel
were free. Pass a solver-backed `deliver` and the refusals appear as
`unassigned` with a `time` reason.

This is the same shape as the finding `docs/capacity-finding.md` exists to
disown — a number produced by machinery that was not measuring what it appears
to measure. The §8.3 check in the same table says `delivery` binds on every one
of these days, which is the honest signal; the delivered total is not.